In [1]:
import h5py
import numpy as np
import torch
import os
from tqdm import tqdm
from pathlib import Path
from scipy import ndimage as ndi

from RAFT.core.raft import RAFT


class DotDict(dict):
    def __getattr__(self, key):
        return self[key]

    def __setattr__(self, key, value):
        self[key] = value


ROOT = Path.cwd()
if not (ROOT / "RAFT").exists() and (ROOT.parent / "RAFT").exists():
    ROOT = ROOT.parent

WEIGHTS = ROOT / "RAFT" / "models" / "raft-small.pth"

#降噪配置：中值滤波 + 硬阈值 + 小连通域剔除
DENOISE_CFG = {
    "median_size": 3,
    "tau_abs": 0.6,
    "tau_rel": 0.08,
    "min_component_area": 64,
}


# 导入你之前定义的 RAFT 加载和降噪函数 (假设在同一目录下或已定义)
# from raft_utils import load_raft_model, build_valid_sky_mask, hard_denoise_flow
def load_raft_model(weight_path=WEIGHTS, small=True):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    args = DotDict(small=small, mixed_precision=True, alternate_corr=False)
    model = RAFT(args).to(device).eval()

    if not weight_path.exists():
        raise FileNotFoundError(f"Cannot find RAFT weights: {weight_path}")

    try:
        state = torch.load(weight_path, map_location=device, weights_only=True)
    except TypeError:
        state = torch.load(weight_path, map_location=device)

    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]

    if isinstance(state, dict):
        first_key = next(iter(state))
        if first_key.startswith("module."):
            state = {k.replace("module.", "", 1): v for k, v in state.items()}

    model.load_state_dict(state)
    return model, device

def hard_denoise_flow(flow_np, valid_mask, cfg=DENOISE_CFG):
    # flow_np: [H, W, 2]
    u = flow_np[..., 0]
    v = flow_np[..., 1]

    u_med = ndi.median_filter(u, size=cfg["median_size"])
    v_med = ndi.median_filter(v, size=cfg["median_size"])
    mag = np.sqrt(u_med ** 2 + v_med ** 2)

    max_mag = float(mag[valid_mask].max()) if valid_mask.any() else float(mag.max())
    tau = max(cfg["tau_abs"], cfg["tau_rel"] * max_mag)

    keep = valid_mask & (mag >= tau)

    # 去掉零散噪点连通域
    labeled, num = ndi.label(keep)
    if num > 0:
        counts = np.bincount(labeled.ravel())
        small_labels = np.where(counts < cfg["min_component_area"])[0]
        if len(small_labels) > 0:
            keep[np.isin(labeled, small_labels)] = False

    out = np.zeros_like(flow_np, dtype=np.float32)
    out[..., 0][keep] = u_med[keep]
    out[..., 1][keep] = v_med[keep]

    return out, keep, tau


def build_valid_sky_mask(image1_np, image2_np, radius_ratio=0.49, black_threshold=8):
    h, w = image1_np.shape[:2]
    cy, cx = h // 2, w // 2
    r = int(min(h, w) * radius_ratio)

    yy, xx = np.ogrid[:h, :w]
    circle_mask = (yy - cy) ** 2 + (xx - cx) ** 2 <= r ** 2

    valid1 = np.any(image1_np > black_threshold, axis=2)
    valid2 = np.any(image2_np > black_threshold, axis=2)
    return circle_mask & valid1 & valid2


def generate_flow_h5(source_h5, times_dir, output_h5, max_frames=None):
    
    model, device = load_raft_model(small=True)
    
    with h5py.File(source_h5, 'r') as src, h5py.File(output_h5, 'w') as dst:
        
        for group_name in ['trainval', 'test']:
            print(f"\n--- 🚀 正在极速处理 {group_name} 分组 (预分配模式) ---")
            
            # 1. 动态拼接时间戳路径并加载
            time_file_path = os.path.join(times_dir, f'times_{group_name}.npy')
            print(f"读取时间戳: {time_file_path}")
            times = np.load(time_file_path, allow_pickle=True)
            
            src_imgs = src[group_name]['global_images_log']
            total_frames = src_imgs.shape[0]
            
            # 严格对齐校验
            assert total_frames == len(times), f"🚨 致命错误：{group_name} 组的图片数({total_frames})与时间戳数({len(times)})不匹配！"
            
            run_frames = total_frames if max_frames is None else min(max_frames, total_frames)

            # ==========================================
            # 🛡️ 核心改造：一次性开辟连续空间，绝不使用 maxshape 和 resize！
            # ==========================================
            dst_grp = dst.create_group(group_name)
            flow_ds = dst_grp.create_dataset(
                'global_flow_log', 
                shape=(run_frames, 2, 256, 256),   # <--- 直接将形状定死为 run_frames
                dtype='float16', 
                chunks=(1, 2, 256, 256),           # Chunk 大小保持单帧，方便按帧读取
                compression='lzf'
            )
            print(f"✅ 已在硬盘上预分配 {run_frames} 帧的连续存储空间。")
                       
            # 强行塞入第 0 帧占位符
            flow_ds[0] = np.zeros((2, 256, 256), dtype=np.float16)
            
            # ==========================================
            # 🚀 开启“内存大巴车”批量读取模式！彻底碾压 I/O 瓶颈
            # ==========================================
            buffer_size = 1000 # 每次吸入 1000 张图，仅占用约 190MB 内存，对电脑毫无压力
            
            # 外层循环：控制大巴车发车
            for start_idx in tqdm(range(1, run_frames, buffer_size), desc=f"{group_name} 总进度"):
                
                # 计算本次大巴车的终点站
                end_idx = min(start_idx + buffer_size, run_frames)
                
                # 🌟 核心魔法：一次性把这一段的图片全部读进内存数组！
                # 注意：必须从 start_idx - 1 开始读，因为算光流永远需要上一帧！
                img_buffer = src_imgs[start_idx - 1 : end_idx]
                
                # 内层循环：在内存中光速处理这 1000 张图
                for local_i in range(end_idx - start_idx):
                    global_i = start_idx + local_i
                    
                    # 检查时间连续性
                    time_delta = (times[global_i] - times[global_i-1]).total_seconds()
                    
                    if time_delta > 65:
                        # 💥 跨日/数据断层拦截
                        flow_ds[global_i] = np.zeros((2, 256, 256), dtype=np.float16)
                        continue
                        
                    # ⚡ 直接从内存大巴车里拿图，速度是光速，彻底告别硬盘寻址！
                    img1 = img_buffer[local_i]
                    img2 = img_buffer[local_i + 1]
                    
                    # 转 Tensor 并扔进 GPU (这里和你原来一样)
                    t1 = torch.from_numpy(img1).float().unsqueeze(0).to(device)
                    t2 = torch.from_numpy(img2).float().unsqueeze(0).to(device)
                    
                    with torch.no_grad():
                        _, flow_up = model(t1, t2, iters=20, test_mode=True)
                        flow_np = flow_up[0].permute(1, 2, 0).cpu().numpy()
                    
                    # 掩码与降噪
                    img1_np = img1.transpose(1, 2, 0)
                    img2_np = img2.transpose(1, 2, 0)
                    valid_mask = build_valid_sky_mask(img1_np, img2_np)
                    
                    flow_dn_raw, _, _ = hard_denoise_flow(flow_np, valid_mask)
                    flow_dn = flow_dn_raw.transpose(2, 0, 1).astype(np.float16)
                    
                    # 写入光流
                    flow_ds[global_i] = flow_dn

    print(f"\n🎉 宗师级全量光流数据生成完毕！文件结构坚如磐石！已保存至 {output_h5}")
                  


In [3]:
# 执行
if __name__ == '__main__':
    print("启动终极光流生成器...")
    
    # 全部指向你的 output_folder
    generate_flow_h5(
        source_h5='output_folder/2019_dataset_dual_V2.h5', 
        times_dir='output_folder',                      # 传入目录，让代码自己去找 trainval 和 test
        output_h5='output_folder/2019_dataset_flow.h5', # 统一生成一个 H5
        max_frames=None                                 # 100 帧探路开关
    )

启动终极光流生成器...

--- 🚀 正在极速处理 trainval 分组 (预分配模式) ---
读取时间戳: output_folder\times_trainval.npy
✅ 已在硬盘上预分配 93483 帧的连续存储空间。


trainval 总进度:   0%|          | 0/94 [00:00<?, ?it/s]e:\造数据集\RAFT\core\raft.py:99: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.args.mixed_precision):
e:\造数据集\RAFT\core\raft.py:110: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.args.mixed_precision):
e:\造数据集\RAFT\core\raft.py:127: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.args.mixed_precision):
trainval 总进度: 100%|██████████| 94/94 [3:56:41<00:00, 151.08s/it]  



--- 🚀 正在极速处理 test 分组 (预分配模式) ---
读取时间戳: output_folder\times_test.npy
✅ 已在硬盘上预分配 7885 帧的连续存储空间。


test 总进度: 100%|██████████| 8/8 [19:42<00:00, 147.75s/it]


🎉 宗师级全量光流数据生成完毕！文件结构坚如磐石！已保存至 output_folder/2019_dataset_flow.h5
